# Give Me Some Credit EDA

각 컬럼을 어떻게 전처리할지, 그리고 왜 그렇게 정했는지를 표와 그래프로 정리한다.

## 1. 준비

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.config import DATA_SET_PATH

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 변수 정의
TARGET = "SeriousDlqin2yrs"
LATE_COLS = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]
BIN_DICT = {
    "age":                                  [0, 20, 30, 40, 50, 60, 70, 80, float("inf")],
    "RevolvingUtilizationOfUnsecuredLines": [0, 0.1, 0.3, 0.5, 0.7, 1, 2, 10, float("inf")],
    "DebtRatio":                            [0, 0.1, 0.3, 0.5, 0.7, 1, 2, 10, 100, float("inf")],
    "MonthlyIncome":                        [0, 1000, 2500, 5000, 7500, 10000, 20000, 50000, float("inf")],
}
COLOUM_MAPPING_DICT = {
    "SeriousDlqin2yrs":                     "90일 이상 연체 경험 여부 (타겟 변수)",
    "RevolvingUtilizationOfUnsecuredLines": "신용카드 및 개인 신용한도 대출의 총 잔액을 총 신용한도로 나눈 비율 (부동산 담보대출, 자동차 할부 같은 할부 대출은 제외)",
    "age":                                  "대출자 나이 (세)",
    "NumberOfTime30-59DaysPastDueNotWorse": "최근 2년간 30~59일 연체한 횟수 (그보다 심한 연체는 제외)",
    "DebtRatio":                            "월 부채 상환액, 위자료, 생활비를 월 총소득으로 나눈 비율",
    "MonthlyIncome":                        "월 소득",
    "NumberOfOpenCreditLinesAndLoans":      "보유 중인 대출(자동차 할부, 주택담보대출 등 할부 대출) 및 신용한도(신용카드 등) 개수",
    "NumberOfTimes90DaysLate":              "90일 이상 연체한 횟수",
    "NumberRealEstateLoansOrLines":         "주택담보대출 및 부동산 대출 개수 (주택담보 신용한도 포함)",
    "NumberOfTime60-89DaysPastDueNotWorse": "최근 2년간 60~89일 연체한 횟수 (그보다 심한 연체는 제외)",
    "NumberOfDependents":                   "본인을 제외한 부양가족 수 (배우자, 자녀 등)"
}
pd.DataFrame(COLOUM_MAPPING_DICT.items(), columns=["컬럼명", "설명"])

In [ ]:
# 데이터 로드
df = pd.read_csv(DATA_SET_PATH, index_col=0)
BASE_RATE = df[TARGET].mean() * 100
print(f"전체 {len(df):,}건, 전체 평균 부도율 {BASE_RATE:.2f}%")

In [ ]:
def show_default_rate(data, group, title):
    """그룹별 건수와 부도율을 표와 막대그래프로 보여준다."""
    table = data.groupby(group, observed=False)[TARGET].agg(["count", "mean"])
    table.columns = ["건수", "부도율(%)"]
    table["부도율(%)"] = (table["부도율(%)"] * 100).round(2)
    display(table)

    x = range(len(table))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(x, table["부도율(%)"], color="steelblue")
    ax.axhline(BASE_RATE, color="red", linestyle="--", label=f"전체 평균 부도율 {BASE_RATE:.1f}%")
    for i, (n, rate) in enumerate(zip(table["건수"], table["부도율(%)"])):
        ax.text(i, rate, f"{n:,}건", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(table.index.astype(str), rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylabel("부도율(%)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. 전체 개요

In [ ]:
# 결측치 개수와 비율
pd.DataFrame({
    "결측치 개수": df.isna().sum(),
    "결측치 비율(%)": (df.isna().mean() * 100).round(2),
})

In [ ]:
# 기본 통계
df.describe(percentiles=[0.01, 0.5, 0.99]).T

In [ ]:
# 컬럼별 값 분포
for key, value in COLOUM_MAPPING_DICT.items():
    if key in BIN_DICT:
        col = pd.cut(df[key], bins=BIN_DICT[key], include_lowest=True)
    else:
        col = df[key]

    vc = col.value_counts().sort_index()
    table = pd.DataFrame({
        "개수": vc,
        "비율(%)": (vc / len(df) * 100).round(2)
    })
    display(Markdown(f"### {key}\n{value}"))
    display(table)

## 3. age

0세 고객이 있는지, 0 다음으로 작은 나이가 몇 살인지 확인한다.

In [ ]:
df["age"].value_counts().sort_index().head(5).to_frame("건수")

**결론: `age = 0`인 행은 삭제한다.**
- 0세는 대출을 받을 수 없으므로 명백한 오류다.
- 1건뿐이라 삭제해도 영향이 없다. 0 다음으로 작은 나이는 21세다.
- 과제의 연령대 구분(20-34 / 35-54 / 55+)에 들어갈 곳이 없다. 중앙값으로 바꾸면 없던 사람을 특정 연령대에 넣게 된다.

## 4. RevolvingUtilizationOfUnsecuredLines

한도 사용률은 원래 0~1 사이 값이어야 한다. 1을 넘는 값이 실제 고객인지 오류인지 구간별 부도율로 확인한다.

In [ ]:
bins = [0, 0.5, 1, 1.5, 2, 3, 5, 10, 100, float("inf")]
group = pd.cut(df["RevolvingUtilizationOfUnsecuredLines"], bins=bins, include_lowest=True)
show_default_rate(df, group, "한도 사용률 구간별 부도율")

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| 2 이하 | 그대로 | - |
| 2 초과 ~ 100 이하 | 2로 자르기 | - |
| 100 초과 | 중앙값으로 대체 | `is_revolving_outlier` |

- **1 ~ 2**: 부도율이 약 40%로 가장 높다. 한도를 넘겨 쓴 실제 고위험 고객이다.
- **2 ~ 100**: 부도율이 23~33%로 여전히 평균의 3~5배다. 오류가 아니라 실제 고위험 고객이다. 148건뿐이라 2로 잘라 1~2 구간과 합쳐도 방향이 틀리지 않는다.
- **100 초과**: 부도율이 4.9%로 평균보다 낮다. 한도를 100배 넘게 쓴 사람이 평범한 고객보다 안전할 수는 없으므로 오류로 본다.
- 100 초과를 2로 자르면 오류 고객이 "가장 위험한 고객"이 되어 방향이 정반대가 된다. 그래서 중앙값으로 바꾸고 표시 컬럼을 추가한다.

## 5. MonthlyIncome

소득이 결측이거나 0, 1처럼 비정상적으로 작은 고객의 부도율을 정상 고객과 비교한다.

In [ ]:
income = df["MonthlyIncome"]
display(income[income <= 10].value_counts().sort_index().to_frame("건수"))

labels = ["정상 (10 초과)", "2 ~ 10", "1", "0", "결측"]
income_group = pd.Series(labels[0], index=df.index)
income_group[income <= 10] = labels[1]
income_group[income == 1] = labels[2]
income_group[income == 0] = labels[3]
income_group[income.isna()] = labels[4]
income_group = pd.Categorical(income_group, categories=labels, ordered=True)
show_default_rate(df, income_group, "월소득 상태별 부도율")

**결론**

| 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- |
| 10 초과 | 그대로 | - |
| 0 ~ 10 | 중앙값으로 대체 | `is_income_invalid` |
| NaN | 중앙값으로 대체 | `is_income_missing` |

- 월소득 0 ~ 10은 실제 소득으로 볼 수 없다. "소득 없음" 또는 "모름"을 뜻하는 임시값으로 본다.
- 0과 1은 부도율(4.0%, 2.8%)이 둘 다 정상 고객(7.0%)보다 낮은 방향이라 한 집단으로 묶는다. 2 ~ 10은 14건뿐이라 같은 집단에 넣는다.
- 결측 고객의 부도율(5.6%)도 정상 고객과 다르다. "소득을 적지 않았다"는 사실 자체가 정보이므로 표시 컬럼으로 남긴다.
- 분포가 치우쳐 있어 평균 대신 중앙값을 쓴다.

## 6. DebtRatio

DebtRatio는 "월 부채 상환액 ÷ 월소득"이다. 소득 상태에 따라 값이 어떻게 달라지는지 확인한다.

In [ ]:
income = df["MonthlyIncome"]
pct = [0.5, 0.9, 0.99, 0.999]
pd.DataFrame({
    "전체":         df["DebtRatio"].describe(percentiles=pct),
    "소득 10 초과": df.loc[income > 10, "DebtRatio"].describe(percentiles=pct),
    "소득 0 ~ 10":  df.loc[income <= 10, "DebtRatio"].describe(percentiles=pct),
    "소득 결측":    df.loc[income.isna(), "DebtRatio"].describe(percentiles=pct),
}).round(3)

In [ ]:
# 소득이 있는 행 중 DebtRatio가 가장 큰 10건
df.loc[df["MonthlyIncome"] > 0, ["MonthlyIncome", "DebtRatio"]].nlargest(10, "DebtRatio")

In [ ]:
# 소득이 정상(10 초과)인 행만의 구간별 부도율
normal = df[df["MonthlyIncome"] > 10]
group = pd.cut(normal["DebtRatio"], bins=BIN_DICT["DebtRatio"], include_lowest=True)
show_default_rate(normal, group, "소득 정상 고객의 DebtRatio 구간별 부도율")

**결론**

| MonthlyIncome | DebtRatio | `monthly_debt_payment` (새 컬럼) |
| --- | --- | --- |
| 10 초과 | 99.9% 분위수(약 10)로 자르기 | DebtRatio × MonthlyIncome |
| 0 초과 ~ 10 | 중앙값으로 대체 | DebtRatio × MonthlyIncome |
| 0 또는 NaN | 중앙값으로 대체 | DebtRatio 값 그대로 |

- 크게 튀는 값은 거의 다 소득이 결측이거나 0 ~ 10인 행이다. 소득이 1인 행은 1로 나눴기 때문에 DebtRatio가 사실상 월 부채 상환액(금액)이다.
- 소득이 정상인 행만 보면 최댓값이 약 170, 표준편차가 약 1.3으로 정상에 가깝다.
- 부도율은 1 ~ 2 구간(13.7%)이 가장 높고, 2 이후는 조금 낮아지지만 여전히 평균보다 높다. 그래서 2가 아니라 99.9% 분위수(약 10)에서 잘라 2 ~ 10 구간의 정보를 지킨다.
- 소득이 비정상인 행은 부도율이 정상 고객보다 낮다. 상한값으로 두면 가장 위험한 고객으로 바뀌어 방향이 반대가 되므로 중앙값으로 바꾼다.
- 중앙값은 소득이 정상(10 초과)인 행으로만 계산한다.
- 아래에서 월 상환액을 따로 보존하는 근거를 확인한다.

### 6-1. 월 상환액 컬럼 `monthly_debt_payment`

소득이 0이거나 결측인 행의 DebtRatio는 비율이 아니라 **월 부채 상환액(금액)**으로 추정된다. DebtRatio를 중앙값으로 바꾸면 이 정보가 사라지므로, 금액은 새 컬럼에 따로 담는다.

추정이 맞는지 확인하기 위해, 소득이 정상인 고객의 월 상환액(DebtRatio × 소득)과 소득이 0 / 1 / 결측인 고객의 DebtRatio 규모를 비교한다.

In [ ]:
income = df["MonthlyIncome"]
pct = [0.5, 0.9, 0.99, 0.999]
pd.DataFrame({
    "소득 10 초과: DebtRatio × 소득": (df["DebtRatio"] * income)[income > 10].describe(percentiles=pct),
    "소득 1: DebtRatio":             df.loc[income == 1, "DebtRatio"].describe(percentiles=pct),
    "소득 0: DebtRatio":             df.loc[income == 0, "DebtRatio"].describe(percentiles=pct),
    "소득 결측: DebtRatio":          df.loc[income.isna(), "DebtRatio"].describe(percentiles=pct),
}).round(1)

**결론: `monthly_debt_payment` 컬럼을 추가한다.**

| MonthlyIncome | `monthly_debt_payment` |
| --- | --- |
| 0 초과 | DebtRatio × MonthlyIncome |
| 0 또는 NaN | DebtRatio 값 그대로 |
| 공통 | 99.9% 분위수(약 28,000)로 자르기 |

- 소득이 0 / 1 / 결측인 고객의 DebtRatio 중앙값(약 600 ~ 1,200)이 소득이 정상인 고객의 월 상환액 중앙값(약 1,660)과 같은 규모다. 90%, 99% 분위수도 비슷한 규모라 "금액"이라는 추정을 뒷받침한다.
- 비율(DebtRatio)과 금액(`monthly_debt_payment`)을 다른 컬럼에 담으므로 단위가 섞이지 않는다.
- 소득이 1인 행은 "DebtRatio × 1 = 금액"이라 소득 0 초과 공식으로 똑같이 계산된다.
- 소득이 정상인 행에도 최댓값이 약 48만인 극단값이 있어 99.9% 분위수로 자른다.
- **주의**: 이 컬럼은 DebtRatio를 중앙값으로 바꾸기 **전의** 원래 값으로 계산해야 한다.
- **한계**: 소득이 0이거나 결측인 행의 DebtRatio가 금액이라는 것은 데이터 설명서로 확인된 사실이 아니라 분포 비교를 통한 추정이다.

## 7. NumberOfDependents

부양가족 수 결측이 소득 결측과 관련 있는지 확인한다.

In [ ]:
income_na = df["MonthlyIncome"].isna().rename("소득 결측")
dep_na = df["NumberOfDependents"].isna().rename("부양가족 결측")
display(pd.crosstab(income_na, dep_na, margins=True))

labels = ["둘 다 있음", "소득만 결측", "둘 다 결측"]
missing_group = pd.Series(labels[0], index=df.index)
missing_group[income_na & ~dep_na] = labels[1]
missing_group[income_na & dep_na] = labels[2]
missing_group = pd.Categorical(missing_group, categories=labels, ordered=True)
show_default_rate(df, missing_group, "결측 조합별 부도율")

**결론: 결측은 중앙값(0)으로 대체하고 `is_dependents_missing`을 추가한다.**
- 부양가족 수 결측 3,924건은 모두 소득 결측 안에 포함된다. 같은 입력 항목을 통째로 건너뛴 것으로 보인다.
- 빠진 정보가 많을수록 부도율이 낮아진다 (6.9% → 5.8% → 4.6%). 결측 자체가 정보이므로 표시 컬럼으로 남긴다.
- 부양가족 수는 정수이고 분포가 치우쳐 있어 평균(약 0.76) 대신 중앙값(0)을 쓴다.

## 8. 연체 횟수 컬럼 3개

`NumberOfTime30-59DaysPastDueNotWorse`, `NumberOfTime60-89DaysPastDueNotWorse`, `NumberOfTimes90DaysLate`에 있는 96, 98 값이 무엇인지 확인한다.

In [ ]:
code = df[LATE_COLS].isin([96, 98]).any(axis=1)
display(Markdown(f"96/98 고객 수: {code.sum()}명 ({code.mean() * 100:.2f}%)"))
display(df.loc[code, LATE_COLS].value_counts().to_frame("건수"))

In [ ]:
# 96/98 고객과 나머지 고객 비교
other_cols = [c for c in df.columns if c not in LATE_COLS]
pd.DataFrame({
    "96/98 중앙값":     df.loc[code, other_cols].median(),
    "나머지 중앙값":    df.loc[~code, other_cols].median(),
    "96/98 결측률(%)":  df.loc[code, other_cols].isna().mean() * 100,
    "나머지 결측률(%)": df.loc[~code, other_cols].isna().mean() * 100,
}).round(3)

In [ ]:
late_group = code.map({False: "정상 값", True: "96/98"})
show_default_rate(df, late_group, "연체 코드 여부별 부도율")

**결론: 96/98은 0으로 바꾸고 `is_late_unknown`을 추가한다.**
- 세 컬럼이 항상 동시에 98(264명) 또는 96(5명)이다. 실제 연체 횟수가 아니라 레코드 단위의 특수 코드다.
- 이 고객들은 사용 중인 계좌 0개, 한도 사용률 1.0 고정, 나이 중앙값 29세, 소득이 낮고 결측이 많다. 부도율은 약 55%다.
- 98을 그대로 두면 모델이 "연체 98번"으로 받아들이고, 실제 최댓값(17)보다 훨씬 커서 스케일링이 망가진다.
- 부도 고객이 원래 적으므로 행을 삭제하지 않는다. 위험 신호는 표시 컬럼이 전달한다.
- 연체 컬럼이 0이 되므로 SHAP이 "연체 98회" 같은 틀린 거절 사유를 내놓지 않는다.
- 추심 중인 고객과 신용 이력이 거의 없는 고객이 섞여 있고, 이 데이터로는 둘을 구분할 수 없다. 그래서 씬파일러 판정에는 쓰지 않는다.

## 9. NumberRealEstateLoansOrLines

부동산 대출이 20개 이상인 고객이 실제 값인지 확인한다. 부동산 대출은 전체 대출·신용한도 개수에 포함되므로, 부동산 대출 수가 전체 개수보다 많으면 오류다.

In [ ]:
cols = ["NumberRealEstateLoansOrLines", "NumberOfOpenCreditLinesAndLoans", "MonthlyIncome", "age", TARGET]
display(df.loc[df["NumberRealEstateLoansOrLines"] >= 20, cols].sort_values("NumberRealEstateLoansOrLines", ascending=False))

invalid = (df["NumberRealEstateLoansOrLines"] > df["NumberOfOpenCreditLinesAndLoans"]).sum()
print(f"부동산 대출 수 > 전체 대출·신용한도 수: {invalid}건")

**결론: 그대로 둔다.**
- 전체 데이터에서 부동산 대출 수가 전체 개수보다 많은 행은 0건이다.
- 20개 이상인 고객은 월소득이 대부분 높고 나이가 38~76세다. 부동산 투자자로 보는 것이 자연스럽다.

## 10. 최종 정리

| 컬럼 | 조건 | 처리 | 추가 컬럼 |
| --- | --- | --- | --- |
| `age` | 0 | 행 삭제 | - |
| `RevolvingUtilizationOfUnsecuredLines` | 2 이하 | 그대로 | - |
| | 2 초과 ~ 100 이하 | 2로 자르기 | - |
| | 100 초과 | 중앙값으로 대체 | `is_revolving_outlier` |
| `MonthlyIncome` | 10 초과 | 그대로 | - |
| | 0 ~ 10 | 중앙값으로 대체 | `is_income_invalid` |
| | NaN | 중앙값으로 대체 | `is_income_missing` |
| `DebtRatio` | 소득 10 초과 | 99.9% 분위수(약 10)로 자르기 | - |
| | 소득 0 ~ 10 또는 NaN | 중앙값으로 대체 | (위 소득 표시 컬럼 사용) |
| `monthly_debt_payment` (새 컬럼) | 소득 0 초과 | DebtRatio × MonthlyIncome | - |
| | 소득 0 또는 NaN | DebtRatio 값 그대로 | - |
| | 공통 | 99.9% 분위수(약 28,000)로 자르기 | - |
| `NumberOfDependents` | NaN | 중앙값(0)으로 대체 | `is_dependents_missing` |
| 연체 횟수 컬럼 3개 | 96 / 98 | 0으로 대체 | `is_late_unknown` |
| `NumberRealEstateLoansOrLines` | - | 그대로 | - |
| `NumberOfOpenCreditLinesAndLoans` | - | 그대로 | - |

**공통 규칙**
- 중앙값과 분위수는 모두 Train 데이터에서만 계산하고, Validation/Test에 똑같이 적용한다.
- 중앙값은 정상 값만으로 계산한다.
  - `RevolvingUtilizationOfUnsecuredLines`: 100 이하인 행
  - `MonthlyIncome`, `DebtRatio`: 소득이 10 초과인 행
- `monthly_debt_payment`는 `MonthlyIncome`과 `DebtRatio`를 바꾸기 **전의** 원래 값으로 먼저 계산한다.
- 행을 삭제하는 경우는 `age = 0` 한 건뿐이다.